In [14]:
# ============================================================
# MRB2 — MRBench Multi-Task Pedagogical Critic
# Softened class-weighted CE (alpha = 0.50)
# ============================================================
#
# TECHNICAL REPAIR V2:
#   Builds RoBERTa pair special tokens explicitly as:
#   <s> history </s></s> response </s>
#   This avoids tokenizer-backend API differences.
#
# TECHNICAL REPAIR V3:
#   The custom PreTrainedModel is defined in a real Python
#   module instead of notebook __main__. This avoids the
#   Transformers 5.x Jupyter source-file inspection bug.
#
# Goal:
# Train ONE model to score a tutor response on four MRBench V3
# pedagogical dimensions:
#
#   1. Mistake_Identification
#   2. Mistake_Location
#   3. Providing_Guidance
#   4. Actionability
#
# Labels:
#   No             -> 0
#   To some extent -> 1
#   Yes            -> 2
#
# Later LinTS continuous feature for each task:
#   score = 0.0 * P(No)
#         + 0.5 * P(To some extent)
#         + 1.0 * P(Yes)
#
# IMPORTANT:
# - Input uses ONLY:
#       conversation_history
#       candidate tutor response
# - Ground_Truth_Solution is NOT used.
# - Split is by conversation_id, never by individual tutor response.
# - Official MRBench V3 test set is unlabeled and is NOT used
#   for model selection in MRB2.
#
# Controlled change from MRB1:
#   RoBERTa-base + shared encoder + four 3-class heads
#   robust recent-history + response token budgeting
#   SAME model/split/tokenization/training schedule as MRB1
#   ONLY change: per-task softened inverse-frequency class weights
#   alpha = 0.50
#
# Primary metric:
#   mean of the four task Macro-F1 scores
#
# Secondary:
#   task Accuracy
#   per-class F1
#   confusion matrices
#   multiclass Brier score
#   ECE (calibration)
#
# Kaggle:
# - Fresh notebook
# - GPU: 2 x T4 recommended
# - Add a Kaggle dataset containing:
#       mrbench_v3_devset.json
# - Suggested Kaggle dataset name:
#       mrbench-v3
# ============================================================

import gc
import json
import math
import random
import shutil
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    RobertaConfig,
    RobertaModel,
    RobertaPreTrainedModel,
)
from transformers.modeling_outputs import SequenceClassifierOutput

SEED = 42
MODEL_NAME = "FacebookAI/roberta-base"
MAX_LENGTH = 512

TASKS = [
    "Mistake_Identification",
    "Mistake_Location",
    "Providing_Guidance",
    "Actionability",
]

LABELS = ["No", "To some extent", "Yes"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

RESULT_DIR = Path("/kaggle/working/mrb2_results")
CHECKPOINT_DIR = Path("/kaggle/working/mrb2_checkpoints")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 78)
print("MRB2 CONFIGURATION")
print("=" * 78)
print("Model:", MODEL_NAME)
print("Tasks:", TASKS)
print("Labels:", LABELS)
print("Max length:", MAX_LENGTH)
print("GPU count:", torch.cuda.device_count())
print("Ground-truth solution used as input: False")
print("Official V3 test used for selection: False")

assert torch.cuda.is_available(), "GPU is required."


MRB2 CONFIGURATION
Model: FacebookAI/roberta-base
Tasks: ['Mistake_Identification', 'Mistake_Location', 'Providing_Guidance', 'Actionability']
Labels: ['No', 'To some extent', 'Yes']
Max length: 512
GPU count: 2
Ground-truth solution used as input: False
Official V3 test used for selection: False


In [15]:
# ============================================================
# LOCATE + LOAD MRBENCH V3 DEV SET
# ============================================================

dev_candidates = list(
    Path("/kaggle/input").glob("**/mrbench_v3_devset.json")
)

print("Found V3 dev candidates:")
for p in dev_candidates:
    print(" ", p)

assert len(dev_candidates) == 1, (
    "Expected exactly one mrbench_v3_devset.json under /kaggle/input. "
    f"Found {len(dev_candidates)}."
)

DEV_PATH = dev_candidates[0]

with DEV_PATH.open("r", encoding="utf-8") as f:
    raw_dev = json.load(f)

print("\nConversations:", len(raw_dev))
assert len(raw_dev) == 300, (
    f"Expected 300 V3 dev conversations; found {len(raw_dev)}."
)

required_top = {
    "conversation_id",
    "conversation_history",
    "tutor_responses",
}

for row in raw_dev:
    assert required_top.issubset(row.keys())

# ------------------------------------------------------------
# Flatten one sample per candidate tutor response.
# ------------------------------------------------------------

examples = []

for row in raw_dev:
    conversation_id = str(row["conversation_id"])
    history = str(row["conversation_history"])

    for tutor_name, response_obj in row["tutor_responses"].items():
        assert "response" in response_obj
        assert "annotation" in response_obj

        ann = response_obj["annotation"]

        for task in TASKS:
            assert task in ann, (
                f"Missing {task} in conversation {conversation_id}, "
                f"tutor {tutor_name}"
            )
            assert ann[task] in LABEL2ID, (
                f"Unexpected label {ann[task]!r} for {task}"
            )

        examples.append({
            "conversation_id": conversation_id,
            "tutor_name": str(tutor_name),
            "conversation_history": history,
            "tutor_response": str(response_obj["response"]),
            "labels": [
                LABEL2ID[ann[task]]
                for task in TASKS
            ],
        })

print("Flattened labelled tutor responses:", len(examples))

assert len(examples) == 2476, (
    f"Expected 2,476 labelled responses; found {len(examples)}."
)

print("\nDataset schema audit: PASSED")


Found V3 dev candidates:
  /kaggle/input/datasets/malindus/mrbench-v3/mrbench_v3_devset.json

Conversations: 300
Flattened labelled tutor responses: 2476

Dataset schema audit: PASSED


In [16]:
# ============================================================
# LABEL DISTRIBUTION + MAJORITY BASELINES
# ============================================================

def task_label_counts(rows, task_idx):
    return Counter(
        LABELS[row["labels"][task_idx]]
        for row in rows
    )

print("=" * 78)
print("FULL V3 DEV LABEL DISTRIBUTIONS")
print("=" * 78)

for task_idx, task in enumerate(TASKS):
    counts = task_label_counts(examples, task_idx)
    print(f"\n{task}")
    for label in LABELS:
        count = counts[label]
        print(
            f"  {label:15s}: "
            f"{count:4d} "
            f"({count / len(examples):.2%})"
        )

print("\nMajority baselines on the complete labelled pool:")
for task_idx, task in enumerate(TASKS):
    y = np.asarray(
        [r["labels"][task_idx] for r in examples]
    )
    majority = Counter(y.tolist()).most_common(1)[0][0]
    pred = np.full_like(y, majority)

    print(
        f"{task:24s} "
        f"majority={ID2LABEL[int(majority)]:15s} "
        f"acc={accuracy_score(y, pred):.4f} "
        f"macro_f1={f1_score(y, pred, labels=[0,1,2], average='macro', zero_division=0):.4f}"
    )


FULL V3 DEV LABEL DISTRIBUTIONS

Mistake_Identification
  No             :  370 (14.94%)
  To some extent :  174 (7.03%)
  Yes            : 1932 (78.03%)

Mistake_Location
  No             :  713 (28.80%)
  To some extent :  220 (8.89%)
  Yes            : 1543 (62.32%)

Providing_Guidance
  No             :  566 (22.86%)
  To some extent :  503 (20.32%)
  Yes            : 1407 (56.83%)

Actionability
  No             :  797 (32.19%)
  To some extent :  369 (14.90%)
  Yes            : 1310 (52.91%)

Majority baselines on the complete labelled pool:
Mistake_Identification   majority=Yes             acc=0.7803 macro_f1=0.2922
Mistake_Location         majority=Yes             acc=0.6232 macro_f1=0.2560
Providing_Guidance       majority=Yes             acc=0.5683 macro_f1=0.2416
Actionability            majority=Yes             acc=0.5291 macro_f1=0.2307


In [17]:
# ============================================================
# LEAKAGE-SAFE INTERNAL SPLIT BY conversation_id
# ============================================================
#
# V3 official test is unlabeled. Therefore MRB2 uses an internal
# held-out validation split from the labelled V3 dev set.
#
# Split:
#   80% conversations -> train
#   20% conversations -> validation
#
# Every response from one conversation stays in one side.
# ============================================================

conversation_ids = sorted(
    {row["conversation_id"] for row in examples}
)

train_cids, val_cids = train_test_split(
    conversation_ids,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)

train_cids = set(train_cids)
val_cids = set(val_cids)

assert train_cids.isdisjoint(val_cids)
assert len(train_cids) == 240
assert len(val_cids) == 60

train_rows = [
    row for row in examples
    if row["conversation_id"] in train_cids
]

val_rows = [
    row for row in examples
    if row["conversation_id"] in val_cids
]

print("Train conversations:", len(train_cids))
print("Validation conversations:", len(val_cids))
print("Train responses:", len(train_rows))
print("Validation responses:", len(val_rows))
print("Conversation overlap:", len(train_cids & val_cids))

assert len(train_rows) + len(val_rows) == len(examples)

print("\nSplit label distributions:")

for task_idx, task in enumerate(TASKS):
    train_counts = task_label_counts(train_rows, task_idx)
    val_counts = task_label_counts(val_rows, task_idx)

    print(f"\n{task}")
    for label in LABELS:
        tr = train_counts[label]
        va = val_counts[label]
        print(
            f"  {label:15s} "
            f"train={tr:4d} ({tr / len(train_rows):.2%}) "
            f"val={va:4d} ({va / len(val_rows):.2%})"
        )

split_manifest = {
    "seed": SEED,
    "train_conversation_ids": sorted(train_cids),
    "validation_conversation_ids": sorted(val_cids),
}

with (RESULT_DIR / "mrb2_split_manifest.json").open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(split_manifest, f, indent=2)

print("\nLeakage-safe conversation split: PASSED")

# ------------------------------------------------------------
# MRB2 ONLY: softened inverse-frequency class weights.
#
# For each task:
#   raw_weight_c  = N / (K * count_c)
#   soft_weight_c = raw_weight_c ** alpha
#
# alpha=0.50 is deliberately conservative: it gives the rare
# "To some extent" class more influence without using the full
# inverse-frequency correction.
# ------------------------------------------------------------

CLASS_WEIGHT_ALPHA = 0.50
TASK_CLASS_WEIGHTS = []

print("\nMRB2 softened class weights (alpha=0.50):")

for task_idx, task in enumerate(TASKS):
    counts = np.asarray(
        [
            sum(
                row["labels"][task_idx] == class_id
                for row in train_rows
            )
            for class_id in range(len(LABELS))
        ],
        dtype=np.float64,
    )

    n = counts.sum()
    k = len(LABELS)

    raw_weights = n / (k * counts)
    soft_weights = raw_weights ** CLASS_WEIGHT_ALPHA

    TASK_CLASS_WEIGHTS.append(
        [float(x) for x in soft_weights]
    )

    print(f"\n{task}")
    for label, count, raw_w, soft_w in zip(
        LABELS,
        counts,
        raw_weights,
        soft_weights,
    ):
        print(
            f"  {label:15s} "
            f"count={int(count):4d} "
            f"raw={raw_w:.6f} "
            f"soft={soft_w:.6f}"
        )

assert len(TASK_CLASS_WEIGHTS) == 4
assert all(len(w) == 3 for w in TASK_CLASS_WEIGHTS)

print("\nClass-weight construction: PASSED")


Train conversations: 240
Validation conversations: 60
Train responses: 1981
Validation responses: 495
Conversation overlap: 0

Split label distributions:

Mistake_Identification
  No              train= 304 (15.35%) val=  66 (13.33%)
  To some extent  train= 137 (6.92%) val=  37 (7.47%)
  Yes             train=1540 (77.74%) val= 392 (79.19%)

Mistake_Location
  No              train= 564 (28.47%) val= 149 (30.10%)
  To some extent  train= 188 (9.49%) val=  32 (6.46%)
  Yes             train=1229 (62.04%) val= 314 (63.43%)

Providing_Guidance
  No              train= 457 (23.07%) val= 109 (22.02%)
  To some extent  train= 422 (21.30%) val=  81 (16.36%)
  Yes             train=1102 (55.63%) val= 305 (61.62%)

Actionability
  No              train= 623 (31.45%) val= 174 (35.15%)
  To some extent  train= 301 (15.19%) val=  68 (13.74%)
  Yes             train=1057 (53.36%) val= 253 (51.11%)

Leakage-safe conversation split: PASSED

MRB2 softened class weights (alpha=0.50):

Mistake_Identifi

In [18]:
# ============================================================
# TOKENIZATION — ROBUST PAIRED BUDGETING
# ============================================================
#
# Why the original `truncation="only_first"` can fail:
# If the candidate tutor response by itself is too long to fit
# inside max_length=512, the tokenizer cannot remove enough tokens
# from sequence A (conversation history) and raises:
#
#   "Sequence to truncate too short to respect max_length"
#
# FIX:
# Build the pair manually with an explicit token budget.
#
# Priorities:
#   1. Keep the candidate tutor response as intact as possible.
#   2. If a response is extremely long, keep BOTH its beginning
#      and ending (head + tail), because pedagogical properties can
#      occur in either place.
#   3. Give all remaining tokens to the MOST RECENT conversation
#      history.
#
# RoBERTa pair format requires 4 special tokens:
#   <s> history </s></s> response </s>
#
# MAX_RESPONSE_TOKENS = 256 is only a ceiling. Normal short
# responses use fewer tokens and automatically leave more room
# for recent history.
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_RESPONSE_TOKENS = 256
PAIR_SPECIAL_TOKENS = tokenizer.num_special_tokens_to_add(pair=True)

assert PAIR_SPECIAL_TOKENS == 4, (
    f"Expected RoBERTa pair to use 4 special tokens, "
    f"found {PAIR_SPECIAL_TOKENS}."
)

CONTENT_BUDGET = MAX_LENGTH - PAIR_SPECIAL_TOKENS

assert tokenizer.bos_token_id is not None
assert tokenizer.eos_token_id is not None

print(
    "RoBERTa special-token IDs:",
    {
        "bos": tokenizer.bos_token_id,
        "eos": tokenizer.eos_token_id,
        "pad": tokenizer.pad_token_id,
    },
)

def prepare_for_model(rows):
    prepared = []

    for row in rows:
        prepared.append({
            "conversation_text": (
                "Conversation history:\n"
                + row["conversation_history"]
            ),
            "response_text": (
                "Candidate tutor response:\n"
                + row["tutor_response"]
                + "\n\nEvaluate pedagogical quality:"
            ),
            "labels": row["labels"],
        })

    return prepared

train_prepared = prepare_for_model(train_rows)
val_prepared = prepare_for_model(val_rows)

train_ds = Dataset.from_list(train_prepared)
val_ds = Dataset.from_list(val_prepared)

def encode_no_special(text):
    return tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        verbose=False,
    )["input_ids"]

def head_tail(ids, max_tokens):
    if len(ids) <= max_tokens:
        return ids

    head = max_tokens // 2
    tail = max_tokens - head

    return (
        ids[:head]
        + ids[-tail:]
    )

def encode_pair_with_budget(
    conversation_text,
    response_text,
):
    history_ids = encode_no_special(
        conversation_text
    )

    response_ids_full = encode_no_special(
        response_text
    )

    # Cap only genuinely extreme responses.
    response_ids = head_tail(
        response_ids_full,
        min(
            MAX_RESPONSE_TOKENS,
            CONTENT_BUDGET,
        ),
    )

    # Everything left goes to the most recent conversation history.
    history_budget = (
        CONTENT_BUDGET
        - len(response_ids)
    )

    if history_budget > 0:
        history_ids = history_ids[
            -history_budget:
        ]
    else:
        history_ids = []

    # Build the RoBERTa pair explicitly.
    #
    # Some recent Transformers tokenizer backends expose
    # RobertaTokenizer without build_inputs_with_special_tokens().
    # RoBERTa's pair layout is:
    #
    #   <s> history </s></s> response </s>
    #
    bos_id = tokenizer.bos_token_id
    eos_id = tokenizer.eos_token_id

    assert bos_id is not None
    assert eos_id is not None

    input_ids = (
        [bos_id]
        + history_ids
        + [eos_id, eos_id]
        + response_ids
        + [eos_id]
    )

    attention_mask = [1] * len(input_ids)

    assert len(input_ids) <= MAX_LENGTH

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "history_tokens_kept": len(history_ids),
        "response_tokens_original": len(
            response_ids_full
        ),
        "response_tokens_kept": len(
            response_ids
        ),
    }

# ------------------------------------------------------------
# Diagnostics before Dataset.map
# ------------------------------------------------------------

val_original_lengths = []
val_response_lengths = []
val_packed_lengths = []
val_response_truncated = 0

for row in val_prepared:
    h_ids = encode_no_special(
        row["conversation_text"]
    )
    r_ids = encode_no_special(
        row["response_text"]
    )

    original_pair_len = (
        len(h_ids)
        + len(r_ids)
        + PAIR_SPECIAL_TOKENS
    )

    packed = encode_pair_with_budget(
        row["conversation_text"],
        row["response_text"],
    )

    val_original_lengths.append(
        original_pair_len
    )
    val_response_lengths.append(
        len(r_ids)
    )
    val_packed_lengths.append(
        len(packed["input_ids"])
    )

    if (
        packed["response_tokens_kept"]
        < packed["response_tokens_original"]
    ):
        val_response_truncated += 1

val_original_lengths = np.asarray(
    val_original_lengths
)

val_response_lengths = np.asarray(
    val_response_lengths
)

val_packed_lengths = np.asarray(
    val_packed_lengths
)

print("Validation original pair token length:")
print(
    "  mean:",
    float(val_original_lengths.mean()),
)
print(
    "  median:",
    float(np.median(val_original_lengths)),
)
print(
    "  max:",
    int(val_original_lengths.max()),
)
print(
    "  >512:",
    int((val_original_lengths > 512).sum()),
)

print("\nValidation candidate-response token length:")
print(
    "  mean:",
    float(val_response_lengths.mean()),
)
print(
    "  median:",
    float(np.median(val_response_lengths)),
)
print(
    "  max:",
    int(val_response_lengths.max()),
)
print(
    f"  >{MAX_RESPONSE_TOKENS}:",
    int(
        (
            val_response_lengths
            > MAX_RESPONSE_TOKENS
        ).sum()
    ),
)

print("\nPacked validation length:")
print(
    "  max:",
    int(val_packed_lengths.max()),
)
print(
    "  responses head-tail truncated:",
    val_response_truncated,
)

def tokenize_batch(batch):
    output = {
        "input_ids": [],
        "attention_mask": [],
    }

    for conversation_text, response_text in zip(
        batch["conversation_text"],
        batch["response_text"],
    ):
        packed = encode_pair_with_budget(
            conversation_text,
            response_text,
        )

        output["input_ids"].append(
            packed["input_ids"]
        )

        output["attention_mask"].append(
            packed["attention_mask"]
        )

    return output

train_tok = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "conversation_text",
        "response_text",
    ],
)

val_tok = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "conversation_text",
        "response_text",
    ],
)

assert max(
    len(x)
    for x in train_tok["input_ids"]
) <= MAX_LENGTH

assert max(
    len(x)
    for x in val_tok["input_ids"]
) <= MAX_LENGTH

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("\nRobust paired tokenization: PASSED")


RoBERTa special-token IDs: {'bos': 0, 'eos': 2, 'pad': 1}
Validation original pair token length:
  mean: 337.31111111111113
  median: 375.0
  max: 1353
  >512: 49

Validation candidate-response token length:
  mean: 59.21818181818182
  median: 49.0
  max: 933
  >256: 4

Packed validation length:
  max: 512
  responses head-tail truncated: 4


Map:   0%|          | 0/1981 [00:00<?, ? examples/s]

Map:   0%|          | 0/495 [00:00<?, ? examples/s]


Robust paired tokenization: PASSED


In [19]:
# ============================================================
# MULTI-TASK ROBERTA MODEL — NOTEBOOK-SAFE TRANSFORMERS 5 FIX
# ============================================================
#
# Transformers 5.x inspects the Python source file of custom
# PreTrainedModel subclasses while initializing them.
#
# A class defined directly in Jupyter/Kaggle belongs to
# "__main__", which may not expose a __file__ attribute.
# That causes:
#
#   AttributeError: module '__main__' has no attribute '__file__'
#
# FIX:
# Write the custom model class to a real .py module in
# /kaggle/working and import it from there.
#
# This changes NO architecture or experimental setting.
# ============================================================

import importlib
import sys

MODEL_MODULE_PATH = Path(
    "/kaggle/working/mrb2_multitask_model.py"
)

MODEL_MODULE_SOURCE = '\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom transformers import (\n    RobertaModel,\n    RobertaPreTrainedModel,\n)\nfrom transformers.modeling_outputs import (\n    SequenceClassifierOutput,\n)\n\n\nclass MultiTaskRoberta(RobertaPreTrainedModel):\n    def __init__(self, config):\n        super().__init__(config)\n\n        self.num_tasks = int(\n            getattr(config, "num_tasks", 4)\n        )\n        self.num_classes = int(\n            getattr(config, "num_labels_per_task", 3)\n        )\n\n        task_class_weights = getattr(\n            config,\n            "task_class_weights",\n            None,\n        )\n        if task_class_weights is None:\n            raise ValueError(\n                "config.task_class_weights is required for MRB2"\n            )\n\n        self.register_buffer(\n            "task_class_weights_tensor",\n            torch.tensor(\n                task_class_weights,\n                dtype=torch.float32,\n            ),\n            persistent=False,\n        )\n\n        self.roberta = RobertaModel(\n            config,\n            add_pooling_layer=False,\n        )\n\n        self.dropout = nn.Dropout(\n            config.hidden_dropout_prob\n        )\n\n        self.classifiers = nn.ModuleList([\n            nn.Linear(\n                config.hidden_size,\n                self.num_classes,\n            )\n            for _ in range(self.num_tasks)\n        ])\n\n        self.post_init()\n\n    def forward(\n        self,\n        input_ids=None,\n        attention_mask=None,\n        labels=None,\n        **kwargs,\n    ):\n        outputs = self.roberta(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            **kwargs,\n        )\n\n        pooled = outputs.last_hidden_state[:, 0, :]\n        pooled = self.dropout(pooled)\n\n        logits = torch.stack(\n            [\n                head(pooled)\n                for head in self.classifiers\n            ],\n            dim=1,\n        )\n\n        loss = None\n\n        if labels is not None:\n            task_losses = []\n\n            for task_idx in range(\n                self.num_tasks\n            ):\n                weights = (\n                    self.task_class_weights_tensor[\n                        task_idx\n                    ].to(\n                        device=logits.device,\n                        dtype=torch.float32,\n                    )\n                )\n\n                task_losses.append(\n                    F.cross_entropy(\n                        logits[:, task_idx, :].float(),\n                        labels[:, task_idx],\n                        weight=weights,\n                    )\n                )\n\n            loss = torch.stack(\n                task_losses\n            ).mean()\n\n        return SequenceClassifierOutput(\n            loss=loss,\n            logits=logits,\n            hidden_states=outputs.hidden_states,\n            attentions=outputs.attentions,\n        )\n'

MODEL_MODULE_PATH.write_text(
    MODEL_MODULE_SOURCE,
    encoding="utf-8",
)

if "/kaggle/working" not in sys.path:
    sys.path.insert(
        0,
        "/kaggle/working",
    )

# Ensure rerunning this cell reloads the current source file.
if "mrb2_multitask_model" in sys.modules:
    del sys.modules["mrb2_multitask_model"]

mrb2_multitask_model = importlib.import_module(
    "mrb2_multitask_model"
)

MultiTaskRoberta = (
    mrb2_multitask_model.MultiTaskRoberta
)

print(
    "Custom model module:",
    mrb2_multitask_model.__file__,
)

assert Path(
    mrb2_multitask_model.__file__
).exists()

# ------------------------------------------------------------
# CONFIG + BASE WEIGHTS
# ------------------------------------------------------------

config = RobertaConfig.from_pretrained(
    MODEL_NAME
)

config.mrb_tasks = TASKS
config.mrb_labels = LABELS
config.num_tasks = len(TASKS)
config.num_labels_per_task = len(LABELS)
config.task_class_weights = TASK_CLASS_WEIGHTS
config.class_weight_alpha = CLASS_WEIGHT_ALPHA

model = MultiTaskRoberta.from_pretrained(
    MODEL_NAME,
    config=config,
)

assert model.num_tasks == 4
assert model.num_classes == 3

# ------------------------------------------------------------
# SMALL FORWARD-PASS SANITY CHECK
# ------------------------------------------------------------

sample_batch = data_collator(
    [
        train_tok[i]
        for i in range(min(2, len(train_tok)))
    ]
)

model_device = next(model.parameters()).device

sample_batch = {
    k: (
        v.to(model_device)
        if torch.is_tensor(v)
        else v
    )
    for k, v in sample_batch.items()
}

model.eval()

with torch.no_grad():
    sample_output = model(
        **sample_batch
    )

assert sample_output.logits.shape == (
    len(sample_batch["input_ids"]),
    4,
    3,
), sample_output.logits.shape

print("Model created successfully.")
print("Tasks:", config.mrb_tasks)
print("Labels:", config.mrb_labels)
print("Class-weight alpha:", config.class_weight_alpha)
print(
    "Sanity logits shape:",
    tuple(sample_output.logits.shape),
)

del sample_batch
del sample_output
model.train()

print(
    "\nNotebook-safe custom model initialization: PASSED"
)


Custom model module: /kaggle/working/mrb2_multitask_model.py


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

MultiTaskRoberta LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifiers.{0, 1, 2, 3}.bias   | MISSING    | 
classifiers.{0, 1, 2, 3}.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model created successfully.
Tasks: ['Mistake_Identification', 'Mistake_Location', 'Providing_Guidance', 'Actionability']
Labels: ['No', 'To some extent', 'Yes']
Class-weight alpha: 0.5
Sanity logits shape: (2, 4, 3)

Notebook-safe custom model initialization: PASSED


In [20]:
# ============================================================
# METRICS + CALIBRATION
# ============================================================

def softmax_np(logits):
    shifted = logits - logits.max(
        axis=-1,
        keepdims=True,
    )
    expv = np.exp(shifted)

    return expv / expv.sum(
        axis=-1,
        keepdims=True,
    )

def multiclass_brier(y_true, probs, n_classes=3):
    onehot = np.eye(n_classes)[y_true]
    return float(
        np.mean(
            np.sum(
                (probs - onehot) ** 2,
                axis=1,
            )
        )
    )

def expected_calibration_error(
    y_true,
    probs,
    n_bins=10,
):
    confidence = probs.max(axis=1)
    prediction = probs.argmax(axis=1)
    correctness = (prediction == y_true).astype(float)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for lo, hi in zip(edges[:-1], edges[1:]):
        if hi == 1.0:
            mask = (confidence >= lo) & (confidence <= hi)
        else:
            mask = (confidence >= lo) & (confidence < hi)

        if not np.any(mask):
            continue

        bin_acc = correctness[mask].mean()
        bin_conf = confidence[mask].mean()

        ece += (
            mask.mean()
            * abs(bin_acc - bin_conf)
        )

    return float(ece)

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction

    # logits: [N, 4, 3]
    # labels: [N, 4]
    predictions = logits.argmax(axis=-1)

    metrics = {}
    task_macro_f1s = []

    for task_idx, task in enumerate(TASKS):
        short_name = [
            "mi", "ml", "pg", "ac"
        ][task_idx]

        y_true = labels[:, task_idx]
        y_pred = predictions[:, task_idx]

        acc = accuracy_score(
            y_true,
            y_pred,
        )

        macro = f1_score(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        )

        metrics[f"{short_name}_accuracy"] = acc
        metrics[f"{short_name}_macro_f1"] = macro

        task_macro_f1s.append(macro)

    metrics["mean_macro_f1"] = float(
        np.mean(task_macro_f1s)
    )

    return metrics


In [21]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.10,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    save_total_limit=4,
    load_best_model_at_end=False,

    fp16=True,
    seed=SEED,
    data_seed=SEED,

    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer GPUs:", trainer.args.n_gpu)

effective_batch = (
    training_args.per_device_train_batch_size
    * max(1, trainer.args.n_gpu)
    * training_args.gradient_accumulation_steps
)

print("Effective global train batch:", effective_batch)
print(
    "Approx optimizer steps/epoch:",
    math.ceil(len(train_rows) / effective_batch),
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer GPUs: 2
Effective global train batch: 16
Approx optimizer steps/epoch: 124


In [22]:
# ============================================================
# TRAIN MRB2
# ============================================================

print("\n" + "=" * 78)
print("STARTING MRB2 TRAINING")
print("=" * 78)

trainer.train()

with (RESULT_DIR / "mrb2_log_history.json").open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        trainer.state.log_history,
        f,
        indent=2,
    )

print("\nEpoch validation history:")

for row in trainer.state.log_history:
    if "eval_mean_macro_f1" in row:
        print(
            f"Epoch {row.get('epoch')}: "
            f"mean_macro_f1={row['eval_mean_macro_f1']:.6f} | "
            f"MI={row['eval_mi_macro_f1']:.6f} "
            f"ML={row['eval_ml_macro_f1']:.6f} "
            f"PG={row['eval_pg_macro_f1']:.6f} "
            f"AC={row['eval_ac_macro_f1']:.6f}"
        )



STARTING MRB2 TRAINING


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Mi Accuracy,Mi Macro F1,Ml Accuracy,Ml Macro F1,Pg Accuracy,Pg Macro F1,Ac Accuracy,Ac Macro F1,Mean Macro F1
1,1.337965,0.830181,0.222222,0.152702,0.282828,0.214130,0.616162,0.254167,0.179798,0.161751,0.195687
2,1.016922,0.733534,0.141414,0.086573,0.282828,0.214130,0.616162,0.254167,0.137374,0.080521,0.158848
3,0.954455,0.753186,0.296970,0.205162,0.355556,0.248845,0.616162,0.254167,0.137374,0.080521,0.197174
4,0.919238,0.714112,0.278788,0.193024,0.303030,0.224895,0.616162,0.254167,0.137374,0.080521,0.188152


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Epoch validation history:
Epoch 1.0: mean_macro_f1=0.195687 | MI=0.152702 ML=0.214130 PG=0.254167 AC=0.161751
Epoch 2.0: mean_macro_f1=0.158848 | MI=0.086573 ML=0.214130 PG=0.254167 AC=0.080521
Epoch 3.0: mean_macro_f1=0.197174 | MI=0.205162 ML=0.248845 PG=0.254167 AC=0.080521
Epoch 4.0: mean_macro_f1=0.188152 | MI=0.193024 ML=0.224895 PG=0.254167 AC=0.080521


In [23]:
# ============================================================
# MANUAL CHECKPOINT EVALUATION
# ============================================================

checkpoint_dirs = sorted(
    CHECKPOINT_DIR.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]),
)

print("Checkpoints:")
for ckpt in checkpoint_dirs:
    print(" ", ckpt)

assert len(checkpoint_dirs) == 4, (
    f"Expected 4 checkpoints, found {len(checkpoint_dirs)}."
)

all_results = []

for epoch_idx, ckpt in enumerate(
    checkpoint_dirs,
    start=1,
):
    print("\n" + "=" * 78)
    print(f"EPOCH {epoch_idx} — {ckpt.name}")
    print("=" * 78)

    ckpt_config = RobertaConfig.from_pretrained(ckpt)
    ckpt_model = MultiTaskRoberta.from_pretrained(
        ckpt,
        config=ckpt_config,
    )

    eval_args = TrainingArguments(
        output_dir=str(
            RESULT_DIR / f"eval_epoch_{epoch_idx}"
        ),
        per_device_eval_batch_size=16,
        fp16=True,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
    )

    eval_trainer = Trainer(
        model=ckpt_model,
        args=eval_args,
        eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    pred_output = eval_trainer.predict(val_tok)

    logits = pred_output.predictions
    true_ids = pred_output.label_ids
    pred_ids = logits.argmax(axis=-1)
    probs = softmax_np(logits)

    result = {
        "epoch": epoch_idx,
        "checkpoint": str(ckpt),
        "tasks": {},
    }

    task_macro_f1s = []

    for task_idx, task in enumerate(TASKS):
        y_true = true_ids[:, task_idx]
        y_pred = pred_ids[:, task_idx]
        task_probs = probs[:, task_idx, :]

        accuracy = float(
            accuracy_score(y_true, y_pred)
        )

        macro = float(
            f1_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            )
        )

        class_f1 = f1_score(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            average=None,
            zero_division=0,
        )

        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1, 2],
        )

        report = classification_report(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            target_names=LABELS,
            output_dict=True,
            digits=6,
            zero_division=0,
        )

        brier = multiclass_brier(
            y_true,
            task_probs,
            n_classes=3,
        )

        ece = expected_calibration_error(
            y_true,
            task_probs,
            n_bins=10,
        )

        # Future LinTS scalar score:
        # 0*P(No) + .5*P(partial) + 1*P(Yes)
        continuous_scores = (
            0.5 * task_probs[:, 1]
            + 1.0 * task_probs[:, 2]
        )

        task_result = {
            "accuracy": accuracy,
            "macro_f1": macro,
            "per_class_f1": {
                label: float(score)
                for label, score
                in zip(LABELS, class_f1)
            },
            "confusion_matrix": cm.tolist(),
            "classification_report": report,
            "brier_score": brier,
            "ece_10bin": ece,
            "continuous_score_mean": float(
                continuous_scores.mean()
            ),
            "continuous_score_std": float(
                continuous_scores.std()
            ),
        }

        result["tasks"][task] = task_result
        task_macro_f1s.append(macro)

        print(f"\n{task}")
        print(" Accuracy :", accuracy)
        print(" Macro-F1:", macro)
        print(
            " Per-class F1:",
            task_result["per_class_f1"],
        )
        print(" Brier:", brier)
        print(" ECE:", ece)
        print(" Confusion matrix:")
        print(cm)

    result["mean_macro_f1"] = float(
        np.mean(task_macro_f1s)
    )

    print(
        "\nMEAN 4-TASK MACRO-F1:",
        result["mean_macro_f1"],
    )

    all_results.append(result)

    # Save per-response predictions for this epoch.
    prediction_path = (
        RESULT_DIR
        / f"mrb2_epoch{epoch_idx}_validation_predictions.jsonl"
    )

    with prediction_path.open(
        "w",
        encoding="utf-8",
    ) as f:

        for i, row in enumerate(val_rows):
            record = {
                "conversation_id": row["conversation_id"],
                "tutor_name": row["tutor_name"],
                "true_labels": {
                    task: LABELS[int(true_ids[i, task_idx])]
                    for task_idx, task
                    in enumerate(TASKS)
                },
                "predicted_labels": {
                    task: LABELS[int(pred_ids[i, task_idx])]
                    for task_idx, task
                    in enumerate(TASKS)
                },
                "continuous_scores": {
                    task: float(
                        0.5 * probs[i, task_idx, 1]
                        + probs[i, task_idx, 2]
                    )
                    for task_idx, task
                    in enumerate(TASKS)
                },
                "probabilities": {
                    task: {
                        label: float(
                            probs[i, task_idx, label_idx]
                        )
                        for label_idx, label
                        in enumerate(LABELS)
                    }
                    for task_idx, task
                    in enumerate(TASKS)
                },
            }

            f.write(json.dumps(record) + "\n")

    del eval_trainer
    del ckpt_model
    gc.collect()
    torch.cuda.empty_cache()


Checkpoints:
  /kaggle/working/mrb2_checkpoints/checkpoint-124
  /kaggle/working/mrb2_checkpoints/checkpoint-248
  /kaggle/working/mrb2_checkpoints/checkpoint-372
  /kaggle/working/mrb2_checkpoints/checkpoint-496

EPOCH 1 — checkpoint-124


Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Mistake_Identification
 Accuracy : 0.2222222222222222
 Macro-F1: 0.1527018177312873
 Per-class F1: {'No': 0.2554027504911591, 'To some extent': 0.0, 'Yes': 0.20270270270270271}
 Brier: 0.7454876907229253
 ECE: 0.39149638233762796
 Confusion matrix:
[[ 65   0   1]
 [ 31   0   6]
 [347   0  45]]

Mistake_Location
 Accuracy : 0.2828282828282828
 Macro-F1: 0.21413041831431903
 Per-class F1: {'No': 0.0, 'To some extent': 0.14795918367346939, 'Yes': 0.49443207126948774}
 Brier: 0.8198181699076698
 ECE: 0.3317106639496004
 Confusion matrix:
[[  0 128  21]
 [  0  29   3]
 [  0 203 111]]

Providing_Guidance
 Accuracy : 0.6161616161616161
 Macro-F1: 0.25416666666666665
 Per-class F1: {'No': 0.0, 'To some extent': 0.0, 'Yes': 0.7625}
 Brier: 0.7606182781958595
 ECE: 0.3775715514866993
 Confusion matrix:
[[  0   0 109]
 [  0   0  81]
 [  0   0 305]]

Actionability
 Accuracy : 0.1797979797979798
 Macro-F1: 0.16175085070772233
 Per-class F1: {'No': 0.2459016393442623, 'To some extent': 0.2393509127

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Mistake_Identification
 Accuracy : 0.296969696969697
 Macro-F1: 0.2051620976198659
 Per-class F1: {'No': 0.2698072805139186, 'To some extent': 0.0, 'Yes': 0.345679012345679}
 Brier: 0.8673369325333427
 ECE: 0.44576797003697866
 Confusion matrix:
[[ 63   0   3]
 [ 30   0   7]
 [308   0  84]]

Mistake_Location
 Accuracy : 0.35555555555555557
 Macro-F1: 0.24884547421923084
 Per-class F1: {'No': 0.0, 'To some extent': 0.17177914110429449, 'Yes': 0.574757281553398}
 Brier: 0.7702583882418955
 ECE: 0.29014095940975226
 Confusion matrix:
[[  0 100  49]
 [  0  28   4]
 [  0 166 148]]

Providing_Guidance
 Accuracy : 0.6161616161616161
 Macro-F1: 0.25416666666666665
 Per-class F1: {'No': 0.0, 'To some extent': 0.0, 'Yes': 0.7625}
 Brier: 0.7646672839596945
 ECE: 0.3811309024541065
 Confusion matrix:
[[  0   0 109]
 [  0   0  81]
 [  0   0 305]]

Actionability
 Accuracy : 0.13737373737373737
 Macro-F1: 0.08052101835405566
 Per-class F1: {'No': 0.0, 'To some extent': 0.24156305506216696, 'Yes': 0

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Mistake_Identification
 Accuracy : 0.2787878787878788
 Macro-F1: 0.19302356309183
 Per-class F1: {'No': 0.26526315789473687, 'To some extent': 0.0, 'Yes': 0.3138075313807531}
 Brier: 0.9713759306929564
 ECE: 0.48000598447491427
 Confusion matrix:
[[ 63   0   3]
 [ 29   0   8]
 [317   0  75]]

Mistake_Location
 Accuracy : 0.30303030303030304
 Macro-F1: 0.22489514222527263
 Per-class F1: {'No': 0.0, 'To some extent': 0.15425531914893617, 'Yes': 0.5204301075268817}
 Brier: 0.8253378680285492
 ECE: 0.38582297286601974
 Confusion matrix:
[[  0 122  27]
 [  0  29   3]
 [  0 193 121]]

Providing_Guidance
 Accuracy : 0.6161616161616161
 Macro-F1: 0.25416666666666665
 Per-class F1: {'No': 0.0, 'To some extent': 0.0, 'Yes': 0.7625}
 Brier: 0.7643627390949943
 ECE: 0.3808345481602833
 Confusion matrix:
[[  0   0 109]
 [  0   0  81]
 [  0   0 305]]

Actionability
 Accuracy : 0.13737373737373737
 Macro-F1: 0.08052101835405566
 Per-class F1: {'No': 0.0, 'To some extent': 0.24156305506216696, 'Yes':

In [24]:
# ============================================================
# SELECT + SAVE BEST MRB2 CHECKPOINT
# ============================================================
#
# Selection criterion:
#   highest mean of the four task Macro-F1 values
# ============================================================

best = max(
    all_results,
    key=lambda x: x["mean_macro_f1"],
)

print("=" * 78)
print("MRB2 BEST CHECKPOINT")
print("=" * 78)
print("Best epoch:", best["epoch"])
print("Mean 4-task Macro-F1:", best["mean_macro_f1"])

MRB1_BASELINE_MEAN_MACRO_F1 = 0.5349478828538041
delta_vs_mrb1 = (
    best["mean_macro_f1"]
    - MRB1_BASELINE_MEAN_MACRO_F1
)

print(
    "MRB1 baseline mean Macro-F1:",
    MRB1_BASELINE_MEAN_MACRO_F1,
)
print("Delta vs MRB1:", delta_vs_mrb1)

for task in TASKS:
    task_result = best["tasks"][task]
    print(
        f"{task:24s} "
        f"Accuracy={task_result['accuracy']:.6f} "
        f"Macro-F1={task_result['macro_f1']:.6f} "
        f"Brier={task_result['brier_score']:.6f} "
        f"ECE={task_result['ece_10bin']:.6f}"
    )

best_checkpoint = Path(best["checkpoint"])

best_config = RobertaConfig.from_pretrained(
    best_checkpoint
)

best_model = MultiTaskRoberta.from_pretrained(
    best_checkpoint,
    config=best_config,
)

BEST_DIR = RESULT_DIR / "mrb2_best_model"

best_model.save_pretrained(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)

# Preserve custom model class needed for later reload.
shutil.copy2(
    MODEL_MODULE_PATH,
    BEST_DIR / "mrb2_multitask_model.py",
)

# Save the local inference contract separately.
inference_contract = {
    "model_class": "MultiTaskRoberta",
    "model_module": "mrb2_multitask_model.py",
    "base_model": MODEL_NAME,
    "tasks": TASKS,
    "labels": LABELS,
    "continuous_score_formula": (
        "0.0*P(No) + 0.5*P(To some extent) + 1.0*P(Yes)"
    ),
    "input_contract": {
        "sequence_a": "conversation_history",
        "sequence_b": "candidate_tutor_response",
        "ground_truth_solution_used": False,
        "packing": "manual_recent_history_plus_response_budget",
        "history_policy": "keep_most_recent_tokens",
        "response_policy": "keep_full_if_possible_else_head_tail",
        "max_response_tokens": MAX_RESPONSE_TOKENS,
        "max_length": MAX_LENGTH,
    },
}

with (BEST_DIR / "mrb2_inference_contract.json").open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        inference_contract,
        f,
        indent=2,
    )

summary = {
    "experiment": "MRB2_RoBERTaBase_Multitask_SoftWeightedCE_Alpha050",
    "seed": SEED,
    "model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "tasks": TASKS,
    "labels": LABELS,
    "loss": "mean softened class-weighted cross-entropy over four heads",
    "class_weight_alpha": CLASS_WEIGHT_ALPHA,
    "task_class_weights": TASK_CLASS_WEIGHTS,
    "mrb1_baseline_best_mean_macro_f1": 0.5349478828538041,
    "split": {
        "train_conversations": len(train_cids),
        "validation_conversations": len(val_cids),
        "train_responses": len(train_rows),
        "validation_responses": len(val_rows),
        "conversation_overlap": 0,
    },
    "official_v3_test_used": False,
    "ground_truth_solution_used_as_input": False,
    "epochs": all_results,
    "best_epoch": best["epoch"],
    "best_mean_macro_f1": best["mean_macro_f1"],
    "best_tasks": best["tasks"],
}

with (RESULT_DIR / "mrb2_summary.json").open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(summary, f, indent=2)

archive = shutil.make_archive(
    str(RESULT_DIR / "mrb2_best_model"),
    "gztar",
    root_dir=BEST_DIR,
)

print("\nSaved best model directory:")
print(BEST_DIR)

print("\nSaved best model archive:")
print(archive)

print("\nSaved summary:")
print(RESULT_DIR / "mrb2_summary.json")

print("\nMRB2 COMPLETE")


MRB2 BEST CHECKPOINT
Best epoch: 3
Mean 4-task Macro-F1: 0.19717381421495478
MRB1 baseline mean Macro-F1: 0.5349478828538041
Delta vs MRB1: -0.33777406863884935
Mistake_Identification   Accuracy=0.296970 Macro-F1=0.205162 Brier=0.867337 ECE=0.445768
Mistake_Location         Accuracy=0.355556 Macro-F1=0.248845 Brier=0.770258 ECE=0.290141
Providing_Guidance       Accuracy=0.616162 Macro-F1=0.254167 Brier=0.764667 ECE=0.381131
Actionability            Accuracy=0.137374 Macro-F1=0.080521 Brier=1.400324 ECE=0.715523


Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved best model directory:
/kaggle/working/mrb2_results/mrb2_best_model

Saved best model archive:
/kaggle/working/mrb2_results/mrb2_best_model.tar.gz

Saved summary:
/kaggle/working/mrb2_results/mrb2_summary.json

MRB2 COMPLETE
